# Task 1: Preprocess and Explore the Data

**GMF Investments — Week 9 Challenge**

Load TSLA, BND, and SPY data (Jan 2015 – Jun 2026), clean it, perform EDA, test stationarity, and compute risk metrics.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_loader import combine_close_prices, fetch_all_assets, save_processed_data
from src.eda import (
    plot_daily_returns,
    plot_price_history,
    plot_return_distribution,
    plot_rolling_volatility,
    run_adf_test,
    summarize_adf_results,
)
from src.preprocessing import (
    calculate_sharpe_ratio,
    calculate_var,
    clean_asset_frame,
    compute_daily_returns,
    detect_return_outliers,
)

sns.set_theme(style='whitegrid')
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

## 1. Extract Historical Financial Data

In [ ]:
asset_data = fetch_all_assets()
asset_data = {t: clean_asset_frame(df) for t, df in asset_data.items()}
save_processed_data(asset_data, DATA_DIR)

prices = combine_close_prices(asset_data)
prices.head()

## 2. Data Quality Summary

In [ ]:
quality = []
for ticker, df in asset_data.items():
    quality.append({
        'Ticker': ticker,
        'Rows': len(df),
        'Start': df['Date'].min().date(),
        'End': df['Date'].max().date(),
        'Missing Values': int(df.isna().sum().sum()),
    })
pd.DataFrame(quality)

**Data quality actions taken:**
- Ensured numeric dtypes for OHLCV columns
- Sorted by date and forward-filled minor gaps
- Combined adjusted close prices into a single wide frame for analysis

## 3. Exploratory Data Analysis

In [ ]:
returns = prices.pct_change().dropna()

fig1 = plot_price_history(prices, 'Adjusted Close Prices (2015–2026)')
plt.show()

fig2 = plot_daily_returns(returns, 'Daily Percentage Returns')
plt.show()

fig3 = plot_rolling_volatility(returns, window=30)
plt.show()

fig4 = plot_return_distribution(returns['TSLA'], 'TSLA')
plt.show()

## 4. Outlier Detection

In [ ]:
tsla_outliers = detect_return_outliers(returns['TSLA'])
tsla_outliers.sort_values('ZScore', key=abs, ascending=False).head(10)

## 5. Stationarity Tests (ADF)

In [ ]:
adf_results = [
    run_adf_test(prices['TSLA'], 'TSLA Close'),
    run_adf_test(returns['TSLA'], 'TSLA Daily Returns'),
    run_adf_test(prices['SPY'], 'SPY Close'),
    run_adf_test(returns['BND'], 'BND Daily Returns'),
]
summarize_adf_results(adf_results)

**Interpretation:** Closing prices are typically non-stationary (unit root), while daily returns are usually stationary. Non-stationary price series require differencing (`d` in ARIMA) before modeling.

## 6. Risk Metrics

In [ ]:
risk_rows = []
for ticker in prices.columns:
    r = returns[ticker]
    risk_rows.append({
        'Asset': ticker,
        'VaR (95%)': calculate_var(r),
        'Sharpe Ratio': calculate_sharpe_ratio(r),
        'Mean Daily Return': r.mean(),
        'Daily Volatility': r.std(),
    })
pd.DataFrame(risk_rows)

### Key Insights
- **TSLA** shows strong upward drift with high volatility — high risk / high return profile.
- **BND** exhibits lower daily fluctuations, supporting its role as a stabilizer.
- **SPY** provides diversified market exposure with moderate volatility.
- Returns are more suitable for risk measurement and ARIMA modeling than raw prices.